# langchain导入及初始化

需要```pip install langchain-openai```

In [1]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4o-mini",
                        model_provider="openai",
                        base_url="https://api.openai-proxy.org/v1",
                        api_key="sk-mQnID8fQ2UhcuV5yEnmpMg62MuI4AW5GxFe5UCwUkblt1xT7")

# 带系统提示词的交互

## 应用于人物设置和信息分析

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]

res = model.invoke(messages)
print(res.content)

Ciao!


# 模板提示词

## 设置模板

In [3]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

"text": "hi!"

对应

"user", "{text}"

---

{language}

对应

"language": "Italian"

## 使用模板

### 应用于信息处理

In [4]:
prompt = prompt_template.invoke({"language": "Italian", "text": "hi!"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English into Italian', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})])

In [5]:
response = model.invoke(prompt)
print(response.content)

Ciao!


# func call

## 应用于情况分析与分支处理

### 建立函数

要有参数类型和返回值类型

注释格式：

"""
Add two integers.

Args:
    a: First integer
    b: Second integer
   
"""

In [6]:
def add(a: int, b: int) -> int:
    """Add two integers.

    Args:
        a: First integer
        b: Second integer
    """
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two integers.

    Args:
        a: First integer
        b: Second integer
    """
    return a * b

### 添加函数列表

In [7]:
tools = [add,multiply]

### 绑定函数列表

In [9]:
llm_with_tools = model.bind_tools(tools)

### 并没有使用函数 只是选择函数和参数

In [12]:
query = "What is 3 * 12? Also, what is 11 + 49?"

res = llm_with_tools.invoke(query).tool_calls
res

[{'name': 'multiply',
  'args': {'a': 3, 'b': 12},
  'id': 'call_gWl3nP5OuKUY36FtEzNQnqwq',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': 11, 'b': 49},
  'id': 'call_U0q7oxLNM6Qz9X1NV9SHo2O9',
  'type': 'tool_call'}]

In [16]:
res[0]["name"]

'multiply'

In [17]:
res[0]["args"]["a"]

3

In [18]:
res[0]["args"]["b"]

12

In [22]:
if res[0]["name"] == "multiply":
    a = res[0]["args"]["a"]
    b = res[0]["args"]["b"]
    print(multiply(a, b))

36
